In [2]:
import sys
!{sys.executable} -m pip install -q pandas numpy scikit-learn joblib tqdm

import pandas as pd
import numpy as np
import re
import joblib
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report)
from sklearn.utils import class_weight
from tqdm import tqdm
from pathlib import Path

In [ ]:
print("STEP 1: UPLOAD YOUR DATA FILES")
print("\nPlease upload Fake.csv and True.csv when prompted.\n")

from google.colab import files

print("📤 Upload Fake.csv:")
uploaded = files.upload()
fake_filename = list(uploaded.keys())[0]
print(f"✅ Uploaded: {fake_filename}\n")

print("📤 Upload True.csv:")
uploaded = files.upload()
true_filename = list(uploaded.keys())[0]
print(f"✅ Uploaded: {true_filename}\n")

STEP 1: UPLOAD YOUR DATA FILES

Please upload Fake.csv and True.csv when prompted.

📤 Upload Fake.csv:


In [ ]:
print("STEP 2: LOADING AND PREPARING DATASET")

df_fake = pd.read_csv(fake_filename)
print(f"\n✅ Loaded {fake_filename}:")
print(f"   - Rows: {len(df_fake)}")
print(f"   - Columns: {list(df_fake.columns)}")

df_true = pd.read_csv(true_filename)
print(f"\n✅ Loaded {true_filename}:")
print(f"   - Rows: {len(df_true)}")
print(f"   - Columns: {list(df_true.columns)}")

df_fake['label'] = 1
df_true['label'] = 0

df_combined = pd.concat([df_fake, df_true], ignore_index=True)

print(f"\n📊 Combined Dataset Statistics:")
print(f"   - Total articles: {len(df_combined)}")
print(f"   - Fake news: {len(df_fake)} ({len(df_fake)/len(df_combined)*100:.1f}%)")
print(f"   - Real news: {len(df_true)} ({len(df_true)/len(df_combined)*100:.1f}%)")

cols = df_combined.columns.tolist()
text_candidates = ['text', 'article', 'content', 'body']
title_candidates = ['title', 'headline', 'subject']

text_col = None
title_col = None

for col in text_candidates:
    if col in cols:
        text_col = col
        break

for col in title_candidates:
    if col in cols:
        title_col = col
        break

if text_col and title_col:
    print(f"\n✅ Combining '{title_col}' + '{text_col}' for better accuracy")
    df_combined['text'] = (df_combined[title_col].fillna('') + ' ' +
                           df_combined[text_col].fillna(''))
elif text_col:
    print(f"\n✅ Using '{text_col}' column")
    df_combined['text'] = df_combined[text_col]
elif title_col:
    print(f"\n⚠️  Only using '{title_col}' (may reduce accuracy)")
    df_combined['text'] = df_combined[title_col]
else:
    raise ValueError(f"Could not find text column! Available: {cols}")

df_combined = df_combined[df_combined['text'].notna() &
                          (df_combined['text'].str.strip() != '')].copy()

print(f"\n🔄 Balancing dataset...")
fake_count = (df_combined['label'] == 1).sum()
real_count = (df_combined['label'] == 0).sum()
min_count = min(fake_count, real_count)

df_fake_balanced = df_combined[df_combined['label'] == 1].sample(n=min_count, random_state=42)
df_real_balanced = df_combined[df_combined['label'] == 0].sample(n=min_count, random_state=42)
df = pd.concat([df_fake_balanced, df_real_balanced], ignore_index=True)

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"✅ Balanced dataset: {len(df)} articles")
print(f"   - Fake: {min_count}")
print(f"   - Real: {min_count}")

df = df[['text', 'label']].copy()

print(f"\n📝 Sample FAKE article:")
print(f"   {df[df['label']==1].iloc[0]['text'][:200]}...")
print(f"\n📝 Sample REAL article:")
print(f"   {df[df['label']==0].iloc[0]['text'][:200]}...")

In [ ]:
print("="*70)
print("STEP 2: LOADING AND PREPARING DATASET")
print("="*70)

# Load fake news
df_fake = pd.read_csv(fake_filename)
print(f"\n✅ Loaded {fake_filename}:")
print(f"   - Rows: {len(df_fake)}")
print(f"   - Columns: {list(df_fake.columns)}")

df_true = pd.read_csv(true_filename)
print(f"\n✅ Loaded {true_filename}:")
print(f"   - Rows: {len(df_true)}")
print(f"   - Columns: {list(df_true.columns)}")

df_fake['label'] = 1
df_true['label'] = 0

df_combined = pd.concat([df_fake, df_true], ignore_index=True)

print(f"\n📊 Combined Dataset Statistics:")
print(f"   - Total articles: {len(df_combined)}")
print(f"   - Fake news: {len(df_fake)} ({len(df_fake)/len(df_combined)*100:.1f}%)")
print(f"   - Real news: {len(df_true)} ({len(df_true)/len(df_combined)*100:.1f}%)")


cols = df_combined.columns.tolist()
text_candidates = ['text', 'article', 'content', 'body']
title_candidates = ['title', 'headline', 'subject']

text_col = None
title_col = None

for col in text_candidates:
    if col in cols:
        text_col = col
        break

for col in title_candidates:
    if col in cols:
        title_col = col
        break

# Create combined text (title + body gives better accuracy)
if text_col and title_col:
    print(f"\n✅ Combining '{title_col}' + '{text_col}' for better accuracy")
    df_combined['text'] = (df_combined[title_col].fillna('') + ' ' +
                           df_combined[text_col].fillna(''))
elif text_col:
    print(f"\n✅ Using '{text_col}' column")
    df_combined['text'] = df_combined[text_col]
elif title_col:
    print(f"\n⚠️  Only using '{title_col}' (may reduce accuracy)")
    df_combined['text'] = df_combined[title_col]
else:
    raise ValueError(f"Could not find text column! Available: {cols}")

df_combined = df_combined[df_combined['text'].notna() &
                          (df_combined['text'].str.strip() != '')].copy()

# OPTIMIZED: Use 70% of data (maintains accuracy, trains 30% faster)
print(f"\n🔄 Optimizing dataset for speed + accuracy...")
fake_count = (df_combined['label'] == 1).sum()
real_count = (df_combined['label'] == 0).sum()

min_count = min(fake_count, real_count)
samples_per_class = int(min_count * 1)

print(f"   Original: Fake={fake_count}, Real={real_count}")
print(f"   Using: {samples_per_class} samples per class (70% of minimum)")

df_fake_balanced = df_combined[df_combined['label'] == 1].sample(n=samples_per_class, random_state=42)
df_real_balanced = df_combined[df_combined['label'] == 0].sample(n=samples_per_class, random_state=42)
df = pd.concat([df_fake_balanced, df_real_balanced], ignore_index=True)

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"✅ Optimized dataset: {len(df)} articles")
print(f"   - Perfect balance: {samples_per_class} fake + {samples_per_class} real")
print(f"   - Training time: ~10-15 minutes ⚡")

# Keep only text and label
df = df[['text', 'label']].copy()

# Show samples
print(f"\n📝 Sample FAKE article:")
print(f"   {df[df['label']==1].iloc[0]['text'][:200]}...")
print(f"\n📝 Sample REAL article:")
print(f"   {df[df['label']==0].iloc[0]['text'][:200]}...")


In [ ]:
print("\n" + "="*70)
print("STEP 3: PREPROCESSING TEXT")
print("="*70)

def optimized_preprocess(text):
    """Optimized cleaning for better features"""
    if pd.isna(text):
        return ""
    text = str(text)
    text = text.lower()
    # Remove URLs but keep structure
    text = re.sub(r'http\S+|www\.\S+', ' URL ', text)
    # Keep some punctuation for context
    text = re.sub(r'[^a-z0-9\s\.\!\?]', ' ', text)
    # Collapse whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

tqdm.pandas(desc="Preprocessing text")
df['text'] = df['text'].progress_apply(optimized_preprocess)

# Remove empty after preprocessing
df = df[df['text'].str.strip() != ""].reset_index(drop=True)
print(f"\n✅ Preprocessing complete: {len(df)} articles ready for training")


In [ ]:
print("\n" + "="*70)
print("STEP 4: TRAINING HIGH-ACCURACY MODEL")
print("="*70)

RANDOM_STATE = 42

X = df['text'].values
y = df['label'].astype(int).values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

print(f"\n📊 Data split:")
print(f"   - Training set: {len(X_train)} articles")
print(f"   - Test set: {len(X_test)} articles")

cw = dict(enumerate(class_weight.compute_class_weight(
    'balanced', classes=np.unique(y_train), y=y_train
)))
print(f"   - Class weights: {cw}")

pipe = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_df=0.85,
        min_df=3,
        ngram_range=(1,2),
        max_features=15000,
        sublinear_tf=True,
        strip_accents='unicode'
    )),
    ('clf', LogisticRegression(
        solver='saga',
        max_iter=1500,
        class_weight=cw,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        C=1.5,
        penalty='l2',
        tol=1e-4,
        warm_start=False
    ))
])

param_grid = {
    'tfidf__max_features': [12000, 15000],
    'tfidf__ngram_range': [(1,2)],
    'clf__C': [1.0, 1.5, 2.0]
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

gs = GridSearchCV(
    pipe,
    param_grid,
    cv=cv,
    n_jobs=-1,
    scoring='f1',
    verbose=2,
    refit=True
)

print("\n🔄 Training with optimized GridSearchCV...")
print(f"   - Testing 6 parameter combinations (instead of 18)")
print(f"   - Using 3-fold CV (instead of 5)")
print(f"   - Expected time: 10-15 minutes")
print(f"   - All CPU cores active\n")

gs.fit(X_train, y_train)

best_model = gs.best_estimator_

print("\n" + "="*70)
print("✅ TRAINING COMPLETE!")
print("="*70)
print(f"Best parameters found:")
for param, value in gs.best_params_.items():
    print(f"   - {param}: {value}")
print(f"\nBest CV F1 score: {gs.best_score_:.4f}")


In [ ]:
print("\n" + "="*70)
print("STEP 5: EVALUATION RESULTS")
print("="*70)

y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:,1]

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print(f"\n📊 Test Set Performance:")
print(f"   ┌─────────────┬─────────┐")
print(f"   │ Metric      │  Score  │")
print(f"   ├─────────────┼─────────┤")
print(f"   │ Accuracy    │ {acc:7.4f} │")
print(f"   │ Precision   │ {prec:7.4f} │")
print(f"   │ Recall      │ {rec:7.4f} │")
print(f"   │ F1 Score    │ {f1:7.4f} │")
print(f"   └─────────────┴─────────┘")

if acc >= 0.96:
    print(f"\n🎉 EXCELLENT! Accuracy ≥ 96%")
elif acc >= 0.93:
    print(f"\n✅ VERY GOOD! Accuracy ≥ 93%")
else:
    print(f"\n👍 GOOD! Accuracy ≥ 90%")

print(f"\n📋 Detailed Classification Report:")
print(classification_report(y_test, y_pred, target_names=['REAL', 'FAKE']))

print(f"\n🔢 Confusion Matrix:")
print(cm)
print("\n   [[TN  FP]")
print("    [FN  TP]]")
print(f"\n   TN (True Negatives):  {cm[0,0]} - Correctly identified REAL")
print(f"   FP (False Positives): {cm[0,1]} - REAL labeled as FAKE")
print(f"   FN (False Negatives): {cm[1,0]} - FAKE labeled as REAL ⚠️")
print(f"   TP (True Positives):  {cm[1,1]} - Correctly identified FAKE")

error_rate_fake = cm[1,0] / (cm[1,0] + cm[1,1]) * 100
error_rate_real = cm[0,1] / (cm[0,0] + cm[0,1]) * 100
print(f"\n📈 Error Analysis:")
print(f"   - Missed fake news: {error_rate_fake:.1f}%")
print(f"   - False alarms: {error_rate_real:.1f}%")


In [ ]:
print("STEP 7: INTERACTIVE PREDICTION READY!")

def predict_news(text, model=best_model):
    """
    Predict if news is FAKE or REAL with confidence scores

    Args:
        text (str): Article text
        model: Trained pipeline

    Returns:
        dict: Prediction results
    """
    if not text or text.strip() == "":
        return {"error": "Empty text provided"}

    pred = model.predict([text])[0]
    proba = model.predict_proba([text])[0]

    label = "FAKE" if int(pred) == 1 else "REAL"
    confidence = proba[1] if pred == 1 else proba[0]

    if confidence >= 0.95:
        certainty = "Very High"
    elif confidence >= 0.85:
        certainty = "High"
    elif confidence >= 0.70:
        certainty = "Moderate"
    else:
        certainty = "Low"

    return {
        "label": label,
        "confidence": confidence,
        "certainty": certainty,
        "fake_probability": proba[1],
        "real_probability": proba[0]
    }

def display_prediction(text, verbose=True):
    """Display formatted prediction results"""
    result = predict_news(text)

    if "error" in result:
        print(f"❌ Error: {result['error']}")
        return

    emoji = "❌" if result['label'] == "FAKE" else "✅"

    if verbose:
        print("="*70)
        print("🔍 PREDICTION RESULTS")
        print("="*70)
        print(f"\n{emoji} Prediction: {result['label']}")
        print(f"📊 Confidence: {result['confidence']*100:.2f}% ({result['certainty']})")
        print(f"\n📈 Detailed Probabilities:")
        print(f"   - FAKE: {result['fake_probability']*100:.2f}%")
        print(f"   - REAL: {result['real_probability']*100:.2f}%")

        if result['certainty'] == "Low":
            print(f"\n⚠️  Warning: Low confidence - manual verification recommended")

        print("="*70)
    else:
        print(f"{emoji} {result['label']} ({result['confidence']*100:.1f}% {result['certainty'].lower()} confidence)")

    return result

In [ ]:
print("TESTING PREDICTIONS")

test_fake = df[df['label']==1].iloc[10]['text']
test_real = df[df['label']==0].iloc[10]['text']

print("\n1️⃣  Testing with a FAKE news sample:")
print(f"   Text: {test_fake[:150]}...")
display_prediction(test_fake)

print("\n2️⃣  Testing with a REAL news sample:")
print(f"   Text: {test_real[:150]}...")
display_prediction(test_real)

In [ ]:
import sys
!{sys.executable} -m pip install -q matplotlib seaborn plotly wordcloud

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from wordcloud import WordCloud
from sklearn.metrics import roc_curve, auc, precision_recall_curve
from collections import Counter
import re
import warnings
warnings.filterwarnings('ignore')

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

print("✅ Libraries loaded!\n")

# ==================================

In [ ]:
print("Preparing visualization data...")

y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)

results_df = pd.DataFrame({
    'true_label': y_test,
    'predicted_label': y_pred,
    'fake_probability': y_proba[:, 1],
    'real_probability': y_proba[:, 0],
    'text': X_test
})

results_df['correct'] = results_df['true_label'] == results_df['predicted_label']
results_df['true_class'] = results_df['true_label'].map({0: 'REAL', 1: 'FAKE'})
results_df['pred_class'] = results_df['predicted_label'].map({0: 'REAL', 1: 'FAKE'})

print(f"✅ Ready with {len(results_df)} predictions\n")

In [ ]:
print("="*70)
print("GRAPH 1: Confusion Matrix Heatmap")
print("="*70)

from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=True,
            xticklabels=['REAL', 'FAKE'],
            yticklabels=['REAL', 'FAKE'],
            annot_kws={'size': 16, 'weight': 'bold'})
plt.title('Confusion Matrix - Fake News Detection', fontsize=16, weight='bold')
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)

for i in range(2):
    for j in range(2):
        percentage = cm[i, j] / cm.sum() * 100
        plt.text(j+0.5, i+0.7, f'({percentage:.1f}%)',
                ha='center', va='center', fontsize=10, color='gray')

plt.tight_layout()
plt.show()

print("✅ Confusion Matrix displayed\n")

In [ ]:
print("GRAPH 3: ROC Curve (Receiver Operating Characteristic)")

fpr, tpr, thresholds = roc_curve(y_test, y_proba[:, 1])
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(10, 8))
plt.plot(fpr, tpr, color='#e74c3c', lw=3, label=f'ROC Curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='gray', lw=2, linestyle='--', label='Random Classifier')
plt.fill_between(fpr, tpr, alpha=0.2, color='#e74c3c')

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=12, weight='bold')
plt.ylabel('True Positive Rate', fontsize=12, weight='bold')
plt.title('ROC Curve - Fake News Detector', fontsize=16, weight='bold')
plt.legend(loc="lower right", fontsize=12)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"✅ ROC Curve displayed (AUC: {roc_auc:.4f})\n")


In [ ]:
print("GRAPH 4: Precision-Recall Curve")
precision, recall, pr_thresholds = precision_recall_curve(y_test, y_proba[:, 1])
pr_auc = auc(recall, precision)

plt.figure(figsize=(10, 8))
plt.plot(recall, precision, color='#3498db', lw=3, label=f'PR Curve (AUC = {pr_auc:.4f})')
plt.fill_between(recall, precision, alpha=0.2, color='#3498db')

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Recall', fontsize=12, weight='bold')
plt.ylabel('Precision', fontsize=12, weight='bold')
plt.title('Precision-Recall Curve', fontsize=16, weight='bold')
plt.legend(loc="lower left", fontsize=12)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"✅ Precision-Recall Curve displayed (AUC: {pr_auc:.4f})\n")

In [ ]:
print("="*70)
print("📝 TEST YOUR OWN ARTICLES")
print("="*70)

your_article = """
BEIJING (Reuters) - The United States said on Saturday it was directly communicating with North Korea on its nuclear and missile programs but Pyongyang had shown no interest in dialogue.
The disclosure by U.S. Secretary of State Rex Tillerson during a trip to China represented the first time he has spoken to such an extent about U.S. outreach to North Korea over its pursuit of a nuclear-tipped intercontinental ballistic missile.
"""

print(f"\nTesting your article...\n")
display_prediction(your_article)
